In [5]:
from typing import Any, Optional
from gymnasium.spaces import Sequence, Box, Tuple, Text
from gymnasium import spaces
import gymnasium as gym
import numpy as np
from numpy.char import array as chararray
import random

import torch as th
import torch.nn as nn

In [6]:
from stable_baselines3.ppo.policies import MlpPolicy, CnnPolicy
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

In [7]:
# TODO: make everything based off floating points, then round to naturals.
# TODO: normalise the action space 
# TODO: make the action space such that more actions are "valid" manipulations. e.g. not needing to select a particular index rather just selecting from a percentage [0, 1] for the idx.
# TODO: also make actions explicit choices on whether to delete, or insert. 
# suggestions:
# simplify state space, i.e. shorter sequences like of size 10.
# simplify action space, i.e. modify one element only at once, or only short sequences. 
# use autoencoder for the state space
# use an lstm or cnn to process the sequence in the agent

# DONE hand crank the model, see if the action it picks makes sense
# DONE then update the environment, check if the environment is updating properly
# DONE run the model for a few steps if all looks good and see if it approaches the goal.
# DONE write a loop that will run the agent on the environment, see the results rendered.
# DONE perform this on a freshly trained environment.
# DONE to make training more effective, start with a vector of environments with a random initial vector?
# DONE loss is increasing as training happens, learning rate too high?

In [43]:
def bytestrings_fixed(l: int) -> gym.Space:
    # Sequence(Box(low=0, high=255, dtype=int), stack=True)
    # return Text(max_length=l, charset=string.hexdigits)
    return Box(low=0, high=255, shape=(l,1), dtype=np.uint8)

def bytestrings_fixed_flat(l: int) -> gym.Space:
    return Box(low=0, high=255, shape=(l,), dtype=np.uint8)
    
def pad_to(a: np.array, l : int) -> np.array:
    '''Returns an array of length `l`, which is `a` padded to exactly
    `l` elements using zeros.
    '''
    e = np.zeros(shape=(l,), dtype=np.uint8)
    pad_len = min(e.size, a.size)
    e[:pad_len] = a[:pad_len]
    return e

def from_padded(a: np.array, pad_elem: int = 0) -> np.array:
    zero_idxs = np.transpose(np.nonzero(a == pad_elem))
    if zero_idxs.size == 0:
        # all doesn't equal `pad_elem` => there are no padding
        return a
    # slice the array until the first zero
    return a[:zero_idxs[0][0]]

def size_of_padded(a: np.array, pad_elem: int = 0) -> int:
    return from_padded(a, pad_elem).size

In [ ]:
class BitstringEnvFixed(gym.Env):
    def __init__(
            self, 
            init_bytestring: Optional[np.array] = None, 
            max_bytestring_len: int = 30,
            max_replacement_length: int = 2, 
            step_cost: float = 0.01):
        super().__init__()

        # constant part of the state
        if init_bytestring is None:
            # generate a random array of size 20 to 100, if bytestring not supplied
            arr_size = random.randint(3, 10)
            init_bytestring = np.array([random.randint(1, 255) for _ in range(arr_size)], dtype=np.uint8)
        self._init_bytestring = pad_to(init_bytestring, max_bytestring_len).reshape((max_bytestring_len, 1))

        self.max_bytestring_len = max_bytestring_len
        self.max_replacement_length = max_replacement_length
        self.step_cost = step_cost

        # dynamic part of the state
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0

    @property
    def observation_space(self):
        return bytestrings_fixed(self.max_bytestring_len)

    @property
    def action_space(self):
        '''Action space:
            - index \in [0, len(bytestring)): O(L)
            - repl_len \in [0, repl_len): O(R)
            - repl_seq \in C ^ repl_len: O(256 ^ repl_len) = O(256 ^ R)
        where L is the max length of the sequence, R is the max length of the replacement
        '''
        index_space = Box(0, size_of_padded(self.bytestring), dtype=int)
        replacement_len_space = Box(0, self.max_replacement_length, dtype=int)
        return Tuple(spaces=[
            index_space, replacement_len_space, bytestrings_fixed_flat(self.max_replacement_length)
            ])

    def reset(self, seed=None, options=None) -> tuple[np.array, dict[str, Any]]:
        super().reset(seed=seed, options=options)
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0
        return self.bytestring, {}  # empty info dict

    def utility(self) -> float:
        '''Evaluates the current state.
        Abstract method. 
        Should be continuous in the sequence, and measure the distance of the current
        value to the "ideal" one.
        '''
        return 0.0

    def step(self, action) -> tuple[np.array, float, bool, bool, dict[str, Any]]:
        '''Performs the action on the state: a string replacement at the chosen index.
        '''
        # applying action on the state
        repl_start = action[0][0]
        repl_end = repl_start + action[1][0]
        repl_str = from_padded(action[2])
        self.bytestring = pad_to(
              np.concatenate([
                self.bytestring[:repl_start, 0], repl_str, self.bytestring[repl_end:, 0]])
            , self.max_bytestring_len).reshape((self.max_bytestring_len, 1))

        # Reward based on the utility. Each step has a default negative punishment.
        current_utility = self.utility()
        reward = 0 - current_utility - self.step_cost
        self.prev_utility = current_utility

        terminated = current_utility == 0
        truncated = False

        return (
            self.bytestring,
            reward,
            terminated, # bool
            truncated, # bool
            {}, # extra info, dict
        )

    def render(self) -> None:
        # print string representing the environment
        print(from_padded(self.bytestring[:, 0]))

    def close(self):
        pass

In [49]:
class HammingEnv(BitstringEnvFixed):
    @staticmethod
    def hamming_d(str_a: np.array, str_b: np.array) -> float:
        length_d = abs(str_a.size - str_b.size)
        compare_upto = str_a.size
        if length_d != 0:
            compare_upto = min(str_a.size, str_b.size)

        mismatches = 0
        for i in range(compare_upto):
            mismatches += 1 if str_a[i] != str_b[i] else 0

        return length_d + mismatches

    def utility(self) -> float:
        '''Distance function: dist(str_a, str_b) = | len(a) - len(b) | + | mismatches |
        '''
        target = np.array([31, 41, 59, 26, 54, 27, 171], dtype=np.uint8)
        source = from_padded(self.bytestring)
        return HammingEnv.hamming_d(source, target)

In [50]:
class FlattenAction(gym.ActionWrapper):
    """Action wrapper that flattens the action."""
    def __init__(self, env):
        super(FlattenAction, self).__init__(env)
        self.action_space = gym.spaces.utils.flatten_space(self.env.action_space)
        
    def action(self, action):
        return gym.spaces.utils.unflatten(self.env.action_space, action)

    def reverse_action(self, action):
        return gym.spaces.utils.flatten(self.env.action_space, action)

In [69]:
class ReshapeChannelFirst(gym.ObservationWrapper):
    def __init__(self, env):
        '''Assumes original observation is a Box of shape (n, 1)'''
        super(ReshapeChannelFirst, self).__init__(env)
        obs = self.observation_space
        (n, _), dtype, low, high = obs.shape, obs.dtype, obs.low.T, obs.high.T
        self.observation_space = Box(low=low, high=high, shape=(1, n), dtype=dtype)

    def observation(self, observation: np.array):
        '''Assumes observation is a 1d array of shape (length, 1)'''
        size = observation.size
        return observation.reshape((1, size))
    
    def reverse_observation(self, t_observation: np.array):
        '''Again assumes original observation has shape (size, 1)'''
        size = t_observation.size
        return t_observation.reshape((size, 1))

In [70]:
env = Monitor(
    gym.wrappers.TimeLimit(
    FlattenAction(
    ReshapeChannelFirst(
    HammingEnv(init_bytestring=np.array([8] * 6, dtype=np.uint8)))),
    100
    ))
check_env(env)

/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:462: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(
/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:473: UserWarning: Your action space has dtype int64, we recommend using np.float32 to avoid cast errors.
  warnings.warn(


In [ ]:
wrapped_env_ctor = lambda **kwargs: gym.wrappers.TimeLimit(
                                        FlattenAction(
                                        ReshapeChannelFirst(
                                        HammingEnv(**kwargs))), 10)
vec_env = make_vec_env(wrapped_env_ctor, n_envs=3)

In [ ]:
class CustomCNN(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the number of unit for the last layer.
    """

    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # We assume CxHxW images (channels first)
        # Re-ordering will be done by pre-preprocessing or wrapper
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            # original architecture on 
            # https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html#on-policy-algorithms
            nn.Conv1d(n_input_channels, 8, kernel_size=3, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv1d(8, 16, kernel_size=2, stride=1, padding=0),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with th.no_grad():
            n_flatten = self.cnn(
                th.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        return self.linear(self.cnn(observations))

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=128),
)

In [ ]:
model = PPO(
    CnnPolicy, 
    vec_env, 
    verbose=1, 
    learning_rate=3e-4,
    policy_kwargs=policy_kwargs,
    # policy_kwargs=dict(net_arch=[64, 64]), 
    tensorboard_log="./logs/ppo_mlp_int_repr")
model.learn(total_timesteps=100_000, tb_log_name="shortseq_envs3_cnnSmall_rate3e-4_timelimit10")

Using cpu device
Box(0, 255, (1, 10), uint8)
Logging to ./logs/ppo_mlp_int_repr/shortseq_envs3_arch64_rate3e-4_timelimit10_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | -81      |
| time/              |          |
|    fps             | 499      |
|    iterations      | 1        |
|    time_elapsed    | 12       |
|    total_timesteps | 6144     |
---------------------------------


In [58]:
def apply_action(obs: np.array, action: np.array) -> np.array:
    action_int = np.floor(action).astype(int) # floor is used instead of round
    mut_idx, mut_substr_length = action_int[:2]
    mut_replacement = from_padded(action_int[2:])
    mut_obs = pad_to(
          np.concatenate([obs[:mut_idx], mut_replacement, obs[mut_idx+mut_substr_length:]])
        , 1000)
    return mut_obs

In [41]:
ob, _ = env.reset()
print("initial:")
env.render()

for i in range(100):
    act, _ = model.predict(ob)
    ob_alt = apply_action(ob, act)
    ob, rwd, trm, tnc, _ = env.step(act)
    print(f"step {i+1}: applied {act=}, reward {rwd=}")

    all_equal = np.all(ob_alt == ob)
    print(f"{all_equal=}")
    if not all_equal:
        print(f"alt: {from_padded(ob_alt)}")
        print(f"obs: {from_padded(ob)}")

initial:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
step 1: applied act=array([0.        , 0.        , 0.        , 0.72586042, 0.0807665 ,
       1.14477146, 0.        , 1.14837575, 0.        , 0.59806156,
       0.        , 1.00162375, 0.16474172, 0.        , 0.        ,
       0.35937113, 0.        , 0.60198927, 0.        , 0.62705624,
       0.11798391, 0.03065547]), reward rwd=-54.01
all_equal=np.True_
step 2: applied act=array([0.        , 0.7042796 , 0.        , 0.44226462, 0.        ,
       0.        , 0.        , 0.        , 0.12762056, 1.71513283,
       1.56543291, 0.24893397, 0.86555594, 0.        , 0.        ,
       0.57110697, 0.        , 0.        , 0.        , 1.18702006,
       0.7562784 , 1.16739738]), reward rwd=-54.01
all_equal=np.True_
step 3: applied act=array([0.        , 0.        , 0.93302208, 0.        , 1.37412715,
       0.        , 0.        , 0.        , 2.28555226, 2.6083262 ,
       0

In [ ]:
target_arr = pad_to(np.array([1] * 40 + [2] * 15), 1000)
obs_init = pad_to(np.array([7] * 30 + [1] * 5 + [2] * 10 + [3] * 10 + [5] * 9), 1000)
obs_2 = pad_to(np.array([3] * 40 + [2] * 15), 1000)
obs_3 = pad_to(np.array([1] * 39 + [100] + [2] * 15), 1000)

action, _ = model.predict(obs_3)
mut_idx, mut_substr_length = action[:2]
mut_addition = action[2:]
print(f"{mut_idx=} {mut_substr_length=}")
print(f"{mut_addition=}")

new_obs = apply_action(obs_3, action)
print(f"obs_3   = {from_padded(obs_3)}")
print(f"new_obs = {from_padded(new_obs)}")

new_obs_real, rwd, term, trnc, info = env.step(action)
print(f"new_obs_env = {from_padded(new_obs_real)}")


mut_idx=np.float64(0.0) mut_substr_length=np.float64(0.0)
mut_addition=array([0.        , 0.43085319, 0.        , 0.81712788, 0.        ,
       0.        , 2.40206909, 0.        , 2.51411057, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])
obs_3   = [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
new_obs = [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
new_obs_env = [2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [ ]:
print(f"input obs: {from_padded(obs_3)}")
env.render()
action, _ = model.predict(obs_3)
print(f"applying: {action=}")
obs, rew, trm, tnc, _ = env.step(action)
env.render()

input obs: [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
[  1   1   2   1   1   1   1   1   1   1 173 173 221 248 159  54 219 242
  31 162  33   4  90  74 190 233 245  48 207 199 126  13 139 178 209 218
 169 160 147   2  71  63 214 254 193 213  86 100 225  36 222  47 149   3
 181  53 208  81 252  86 211 182 151  97 111 176 141  98 132 175  43  20
 151  36 168 218  74 239 210  74  35  97 207 256   2 107 155  29  53 240
  96 182 230 153 180  75  50  71  47 243 116  53  91 119 123 231  76   1
   1   1   1   1   1   1   1 204  23 128  22  49  78 102 199  19  55  41
  80  15 220 157  59 135 135 109 200   2   2   2   2   2   2   2   2   2
   2   2]
applying: action=array([0.3663674 , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       1.41130352, 

In [39]:
evaluate_policy(model, env)

(np.float64(-5401.0), np.float64(0.0))

In [ ]:
a1 = pad_to(np.array([7] * 30 + [1] * 5 + [2] * 10 + [3] * 10 + [5] * 9), 1000)
a2 = pad_to(np.array([1] * 40 + [2] * 15), 1000)
HammingEnv.hamming_d(a1, a2)

54